# 🏦 Loan Approval Analysis — Complete Workflow

**Author:** Fredys Caballero — BI Analyst  
**Dataset:** Loan Prediction Dataset (614 applications)  
**Source:** https://github.com/dphi-official/Datasets  

---

## 🎯 Objective

End-to-end analysis of a loan approval dataset to understand which factors drive approval decisions, identify risk segments, and extract actionable business insights.

## 📋 Workflow

| Step | Description |
|------|-------------|
| **1. Setup** | Libraries and configuration |
| **2. Load** | Read data from URL |
| **3. Explore** | Understand structure, types, and distributions |
| **4. Clean** | Handle nulls, duplicates, and types |
| **5. Engineer** | Create financial features (DTI, risk score, etc.) |
| **6. Analyze** | Descriptive statistics and statistical tests |
| **7. Visualize** | Dashboard of 8 charts |
| **8. Insights** | Key findings and business recommendations |

---

**Key questions answered:**
- What is the overall approval rate and how does it vary by segment?
- Which variables have the strongest relationship with approval?
- Is the difference in income between approved and rejected applicants statistically significant?
- Which property area and education level has the highest approval rate?

---
## 📦 Step 1 — Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Color palette
GREEN  = '#27ae60'
RED    = '#e74c3c'
BLUE   = '#2980b9'
ORANGE = '#e67e22'
GRAY   = '#7f8c8d'

print('✅ Setup complete')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

---
## 📥 Step 2 — Load Data

In [ ]:
URL = 'https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv'

try:
    df_raw = pd.read_csv(URL)
    print('✅ Loaded from URL')
except Exception:
    # Fallback: create a representative sample dataset
    print('⚠️  URL unavailable — generating sample dataset')
    np.random.seed(42)
    n = 614
    df_raw = pd.DataFrame({
        'Loan_ID'           : [f'LP{str(i).zfill(6)}' for i in range(1, n+1)],
        'Gender'            : np.random.choice(['Male','Female',None], n, p=[.79,.19,.02]),
        'Married'           : np.random.choice(['Yes','No'], n, p=[.65,.35]),
        'Dependents'        : np.random.choice(['0','1','2','3+',None], n, p=[.34,.24,.19,.19,.04]),
        'Education'         : np.random.choice(['Graduate','Not Graduate'], n, p=[.78,.22]),
        'Self_Employed'     : np.random.choice(['No','Yes',None], n, p=[.81,.13,.06]),
        'ApplicantIncome'   : np.random.gamma(4, 1200, n).astype(int),
        'CoapplicantIncome' : np.round(np.random.exponential(1000, n), 2),
        'LoanAmount'        : np.round(np.random.gamma(3, 50, n)),
        'Loan_Amount_Term'  : np.random.choice([360,180,240,300,480], n, p=[.80,.10,.05,.03,.02]),
        'Credit_History'    : np.random.choice([1.0, 0.0, None], n, p=[.84,.08,.08]),
        'Property_Area'     : np.random.choice(['Urban','Semiurban','Rural'], n, p=[.38,.35,.27]),
        'Loan_Status'       : np.random.choice(['Y','N'], n, p=[.69,.31])
    })

print(f'\n📊 Dataset loaded')
print(f'   Shape   : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'   Columns : {list(df_raw.columns)}')

df_raw.head(8)

---
## 🔍 Step 3 — Exploratory Data Analysis

In [ ]:
print('=' * 60)
print('STRUCTURE & TYPES')
print('=' * 60)
print()
df_raw.info()

In [ ]:
print('=' * 60)
print('SUMMARY STATISTICS')
print('=' * 60)
df_raw.describe(include='all')

In [ ]:
print('=' * 60)
print('MISSING VALUES')
print('=' * 60)

null_report = pd.DataFrame({
    'Missing'   : df_raw.isnull().sum(),
    'Percentage': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
}).query('Missing > 0').sort_values('Missing', ascending=False)

if len(null_report) > 0:
    print()
    print(null_report.to_string())
    print(f'\n  Columns affected : {len(null_report)}')
    print(f'  Total null cells : {null_report["Missing"].sum()}')
else:
    print('  ✅ No missing values')

In [ ]:
print('=' * 60)
print('TARGET DISTRIBUTION — Loan_Status')
print('=' * 60)

target = df_raw['Loan_Status'].value_counts()
target_pct = (df_raw['Loan_Status'].value_counts(normalize=True) * 100).round(1)

summary = pd.DataFrame({'Count': target, 'Percentage': target_pct})
summary.index = summary.index.map({'Y': 'Approved', 'N': 'Rejected'})
print()
print(summary.to_string())

print()
print('Categorical column distributions:')
cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Dependents']
for col in cat_cols:
    print(f'\n  {col}:')
    vc = df_raw[col].value_counts(dropna=False)
    for val, cnt in vc.items():
        pct = cnt / len(df_raw) * 100
        print(f'    {str(val):<15} {cnt:>4}  ({pct:.1f}%)')

---
## 🧹 Step 4 — Data Cleaning

In [ ]:
df = df_raw.copy()

print('Cleaning strategy:')
print('  Numeric  nulls → fill with median (robust to outliers)')
print('  Category nulls → fill with mode   (most frequent value)')
print()

# ── Numeric columns: fill with median ────────────────────────────────────────
for col in ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']:
    n_nulls = df[col].isnull().sum()
    if n_nulls > 0:
        fill_val = df[col].median()
        df[col].fillna(fill_val, inplace=True)
        print(f'  ✅ {col:<20}: {n_nulls} nulls → filled with median ({fill_val:.1f})')

# ── Categorical columns: fill with mode ──────────────────────────────────────
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    n_nulls = df[col].isnull().sum()
    if n_nulls > 0:
        fill_val = df[col].mode()[0]
        df[col].fillna(fill_val, inplace=True)
        print(f'  ✅ {col:<20}: {n_nulls} nulls → filled with mode ("{fill_val}")')

# ── Remove duplicates ─────────────────────────────────────────────────────────
dups = df.duplicated().sum()
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f'  ✅ Duplicates removed: {dups}')

# ── Verify ────────────────────────────────────────────────────────────────────
print()
print(f'  Remaining nulls   : {df.isnull().sum().sum()}')
print(f'  Rows before       : {len(df_raw):,}')
print(f'  Rows after        : {len(df):,}')
print('  ✅ Cleaning complete')

---
## ⚙️ Step 5 — Feature Engineering

Creating meaningful financial variables from raw columns.

In [ ]:
df_feat = df.copy()

# ── Financial ratios ─────────────────────────────────────────────────────────
# Total household income (applicant + co-applicant)
df_feat['Total_Income']     = df_feat['ApplicantIncome'] + df_feat['CoapplicantIncome']

# Debt-to-Income ratio — key risk metric in lending
# Higher DTI = higher debt burden relative to income
df_feat['DTI']              = (df_feat['LoanAmount'] / df_feat['ApplicantIncome']).round(4)

# Approximate monthly payment
df_feat['Monthly_Payment']  = (df_feat['LoanAmount'] / (df_feat['Loan_Amount_Term'] / 12)).round(2)

# Income per dependent — purchasing power after family obligations
df_feat['Dependents_Num']   = df_feat['Dependents'].replace({'3+': 3}).astype(int)
df_feat['Income_Per_Dep']   = (df_feat['Total_Income'] / (df_feat['Dependents_Num'] + 1)).round(2)

# ── Risk score (0–100) ────────────────────────────────────────────────────────
# Composite score: credit history (40%) + low DTI (30%) + high income (30%)
df_feat['Risk_Score'] = (
    (df_feat['Credit_History'] * 40) +
    ((1 - df_feat['DTI'].clip(0, 1)) * 30) +
    ((df_feat['Total_Income'] / df_feat['Total_Income'].max()) * 30)
).round(2)

df_feat['Risk_Category'] = pd.cut(
    df_feat['Risk_Score'],
    bins=[0, 40, 70, 100],
    labels=['High Risk', 'Medium Risk', 'Low Risk']
)

# ── Binary encodings for analysis ─────────────────────────────────────────────
df_feat['Is_Approved']      = (df_feat['Loan_Status'] == 'Y').astype(int)
df_feat['Is_Graduate']      = (df_feat['Education']   == 'Graduate').astype(int)
df_feat['Is_Married']       = (df_feat['Married']     == 'Yes').astype(int)
df_feat['Is_Urban']         = (df_feat['Property_Area'] == 'Urban').astype(int)

# ── Summary ───────────────────────────────────────────────────────────────────
new_features = ['Total_Income','DTI','Monthly_Payment','Income_Per_Dep',
                'Risk_Score','Risk_Category','Is_Approved','Is_Graduate',
                'Is_Married','Is_Urban']

print(f'✅ Feature engineering complete')
print(f'   New features created: {len(new_features)}')
for f in new_features:
    print(f'   • {f}')

print()
df_feat[['ApplicantIncome','LoanAmount','Total_Income','DTI','Risk_Score','Risk_Category']].head(6)

---
## 📊 Step 6 — Statistical Analysis

In [ ]:
print('=' * 60)
print('1. APPROVAL RATE BY SEGMENT')
print('=' * 60)

overall_rate = df_feat['Is_Approved'].mean() * 100
print(f'\n  Overall approval rate: {overall_rate:.1f}%')

# By property area
print('\n  By Property Area:')
area_stats = (df_feat.groupby('Property_Area')
              .agg(n=('Is_Approved','count'),
                   approval_rate=('Is_Approved', lambda x: round(x.mean()*100, 1)),
                   avg_income=('ApplicantIncome', lambda x: round(x.mean(), 0)),
                   avg_dti=('DTI', lambda x: round(x.mean(), 3)))
              .sort_values('approval_rate', ascending=False))
print(area_stats.to_string())

# By education
print('\n  By Education:')
edu_stats = (df_feat.groupby('Education')
             .agg(n=('Is_Approved','count'),
                  approval_rate=('Is_Approved', lambda x: round(x.mean()*100, 1)),
                  avg_income=('ApplicantIncome', lambda x: round(x.mean(), 0)))
             .sort_values('approval_rate', ascending=False))
print(edu_stats.to_string())

# By credit history
print('\n  By Credit History:')
credit_stats = (df_feat.groupby('Credit_History')
                .agg(n=('Is_Approved','count'),
                     approval_rate=('Is_Approved', lambda x: round(x.mean()*100, 1)))
                .rename(index={0.0: 'No History', 1.0: 'Good History'}))
print(credit_stats.to_string())

# By risk category
print('\n  By Risk Category:')
risk_stats = (df_feat.groupby('Risk_Category', observed=True)
              .agg(n=('Is_Approved','count'),
                   approval_rate=('Is_Approved', lambda x: round(x.mean()*100, 1)),
                   avg_dti=('DTI', lambda x: round(x.mean(), 3)))
              .sort_values('approval_rate', ascending=False))
print(risk_stats.to_string())

In [ ]:
print('=' * 60)
print('2. STATISTICAL TESTS')
print('=' * 60)

# ── T-Test: Income difference between approved and rejected ───────────────────
print('\n  T-Test: Applicant Income — Approved vs Rejected')

approved   = df_feat[df_feat['Is_Approved'] == 1]['ApplicantIncome']
rejected   = df_feat[df_feat['Is_Approved'] == 0]['ApplicantIncome']
t_stat, p_val = stats.ttest_ind(approved, rejected)

print(f'    Mean (Approved) : ${approved.mean():>10,.0f}')
print(f'    Mean (Rejected) : ${rejected.mean():>10,.0f}')
print(f'    Difference      : ${approved.mean() - rejected.mean():>10,.0f}')
print(f'    t-statistic     : {t_stat:.4f}')
print(f'    p-value         : {p_val:.4f}')
print(f'    Result          : {"✅ Significant difference (p < 0.05)" if p_val < 0.05 else "❌ Not significant"}')

# ── Chi-Square: Education vs Approval ─────────────────────────────────────────
print('\n  Chi-Square: Education vs Loan Status')

ct    = pd.crosstab(df_feat['Education'], df_feat['Loan_Status'])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)

print(ct.to_string())
print(f'\n    Chi²        : {chi2:.4f}')
print(f'    p-value     : {p_chi:.4f}')
print(f'    Result      : {"✅ Significant association (p < 0.05)" if p_chi < 0.05 else "❌ No significant association"}')

# ── Pearson Correlation: Income vs Loan Amount ────────────────────────────────
print('\n  Pearson Correlation: Applicant Income vs Loan Amount')

r, p_r = stats.pearsonr(df_feat['ApplicantIncome'], df_feat['LoanAmount'])
strength = 'Strong' if abs(r) > 0.7 else ('Moderate' if abs(r) > 0.3 else 'Weak')

print(f'    r coefficient   : {r:.4f}')
print(f'    p-value         : {p_r:.6f}')
print(f'    Interpretation  : {strength} positive correlation')

In [ ]:
print('=' * 60)
print('3. CORRELATION MATRIX')
print('=' * 60)

num_cols    = ['ApplicantIncome','CoapplicantIncome','LoanAmount',
               'Loan_Amount_Term','Credit_History','Total_Income',
               'DTI','Risk_Score','Is_Approved']

corr_matrix = df_feat[num_cols].corr()

print('\nCorrelation with Is_Approved (sorted):')
corr_with_target = corr_matrix['Is_Approved'].drop('Is_Approved').sort_values(ascending=False)
for var, val in corr_with_target.items():
    bar   = '█' * int(abs(val) * 20)
    sign  = '+' if val >= 0 else '-'
    print(f'  {var:<22}: {sign}{abs(val):.3f}  {bar}')

---
## 📈 Step 7 — Visualization Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 16))
fig.suptitle('Loan Approval Analysis — Dashboard', fontsize=16, fontweight='bold', y=0.98)

# ── P1: Target distribution ───────────────────────────────────────────────────
ax1 = fig.add_subplot(4, 2, 1)

labels = ['Approved', 'Rejected']
counts = [df_feat['Is_Approved'].sum(), (df_feat['Is_Approved'] == 0).sum()]
colors = [GREEN, RED]
bars   = ax1.bar(labels, counts, color=colors, width=0.5, edgecolor='white')
for bar, cnt in zip(bars, counts):
    pct = cnt / len(df_feat) * 100
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax1.set_title('Approval Distribution', fontweight='bold')
ax1.set_ylabel('Applications')
ax1.set_ylim(0, max(counts) * 1.2)
ax1.grid(axis='y', alpha=0.3)

# ── P2: Approval rate by property area ────────────────────────────────────────
ax2 = fig.add_subplot(4, 2, 2)

rates = df_feat.groupby('Property_Area')['Is_Approved'].mean() * 100
rates = rates.sort_values(ascending=True)
bar_colors = [GREEN if v >= 70 else ORANGE if v >= 60 else RED for v in rates.values]
hbars = ax2.barh(rates.index, rates.values, color=bar_colors, edgecolor='white')
for bar, val in zip(hbars, rates.values):
    ax2.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontweight='bold', fontsize=9)
ax2.axvline(x=70, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label='70% target')
ax2.set_title('Approval Rate by Area', fontweight='bold')
ax2.set_xlabel('Approval Rate (%)')
ax2.set_xlim(0, 100)
ax2.legend(fontsize=8)

# ── P3: Income distribution by status ─────────────────────────────────────────
ax3 = fig.add_subplot(4, 2, 3)

cap = df_feat['ApplicantIncome'].quantile(0.95)
for status, color, label in [('Y', GREEN, 'Approved'), ('N', RED, 'Rejected')]:
    data = df_feat[df_feat['Loan_Status'] == status]['ApplicantIncome'].clip(upper=cap)
    ax3.hist(data, bins=30, alpha=0.65, color=color, label=label, edgecolor='white')
ax3.set_title('Income Distribution by Status', fontweight='bold')
ax3.set_xlabel('Applicant Income ($)')
ax3.set_ylabel('Frequency')
ax3.legend()
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── P4: Boxplot — loan amount by area ─────────────────────────────────────────
ax4 = fig.add_subplot(4, 2, 4)

area_order = df_feat.groupby('Property_Area')['LoanAmount'].median().sort_values().index
data_list  = [df_feat[df_feat['Property_Area'] == a]['LoanAmount'].dropna() for a in area_order]
bp = ax4.boxplot(data_list, labels=area_order, patch_artist=True,
                  medianprops={'color':'white','linewidth':2})
box_colors = [BLUE, GREEN, ORANGE]
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax4.set_title('Loan Amount by Property Area', fontweight='bold')
ax4.set_ylabel('Loan Amount')
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# ── P5: Approval rate by education × area ─────────────────────────────────────
ax5 = fig.add_subplot(4, 2, 5)

edu_area = df_feat.groupby(['Property_Area','Education'])['Is_Approved'].mean().unstack() * 100
x   = np.arange(len(edu_area.index))
w   = 0.35
b1  = ax5.bar(x - w/2, edu_area['Graduate'],     w, label='Graduate',     color=BLUE,   alpha=0.85)
b2  = ax5.bar(x + w/2, edu_area['Not Graduate'], w, label='Not Graduate', color=ORANGE, alpha=0.85)
ax5.axhline(y=70, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax5.set_xticks(x)
ax5.set_xticklabels(edu_area.index)
ax5.set_title('Approval Rate: Education × Area', fontweight='bold')
ax5.set_ylabel('Approval Rate (%)')
ax5.set_ylim(0, 100)
ax5.legend(fontsize=8)

# ── P6: Scatter — income vs loan amount ───────────────────────────────────────
ax6 = fig.add_subplot(4, 2, 6)

sample = df_feat.sample(min(400, len(df_feat)), random_state=42)
for status, color, label in [('Y', GREEN, 'Approved'), ('N', RED, 'Rejected')]:
    sub = sample[sample['Loan_Status'] == status]
    ax6.scatter(sub['ApplicantIncome'].clip(upper=cap),
                sub['LoanAmount'], alpha=0.45, c=color, s=22, label=label)
ax6.set_title('Income vs Loan Amount', fontweight='bold')
ax6.set_xlabel('Applicant Income ($)')
ax6.set_ylabel('Loan Amount')
ax6.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax6.legend(fontsize=8)

# ── P7: DTI by risk category ──────────────────────────────────────────────────
ax7 = fig.add_subplot(4, 2, 7)

risk_order  = ['High Risk', 'Medium Risk', 'Low Risk']
risk_colors = [RED, ORANGE, GREEN]
risk_data   = [df_feat[df_feat['Risk_Category'] == r]['DTI'].clip(upper=2).dropna()
               for r in risk_order]
vp = ax7.violinplot(risk_data, positions=[1,2,3], showmedians=True)
for i, (body, color) in enumerate(zip(vp['bodies'], risk_colors)):
    body.set_facecolor(color)
    body.set_alpha(0.65)
ax7.set_xticks([1,2,3])
ax7.set_xticklabels(risk_order, fontsize=8)
ax7.set_title('DTI by Risk Category', fontweight='bold')
ax7.set_ylabel('Debt-to-Income Ratio')

# ── P8: Correlation heatmap ───────────────────────────────────────────────────
ax8 = fig.add_subplot(4, 2, 8)

corr_cols = ['ApplicantIncome','LoanAmount','Credit_History',
             'Total_Income','DTI','Risk_Score','Is_Approved']
corr_sub  = df_feat[corr_cols].corr()
mask      = np.triu(np.ones_like(corr_sub, dtype=bool))

sns.heatmap(corr_sub, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax8,
            cbar_kws={'shrink': 0.7},
            annot_kws={'size': 7})
ax8.set_title('Correlation Matrix', fontweight='bold')
ax8.tick_params(axis='x', labelsize=7, rotation=45)
ax8.tick_params(axis='y', labelsize=7)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()
print('✅ Dashboard rendered')

---
## 💡 Step 8 — Key Insights & Recommendations

In [ ]:
print('=' * 65)
print('KEY INSIGHTS')
print('=' * 65)

overall     = df_feat['Is_Approved'].mean() * 100
best_area   = df_feat.groupby('Property_Area')['Is_Approved'].mean().idxmax()
best_rate   = df_feat.groupby('Property_Area')['Is_Approved'].mean().max() * 100
grad_rate   = df_feat[df_feat['Education']=='Graduate']['Is_Approved'].mean() * 100
nograd_rate = df_feat[df_feat['Education']=='Not Graduate']['Is_Approved'].mean() * 100
cred_yes    = df_feat[df_feat['Credit_History']==1.0]['Is_Approved'].mean() * 100
cred_no     = df_feat[df_feat['Credit_History']==0.0]['Is_Approved'].mean() * 100
dti_approved= df_feat[df_feat['Is_Approved']==1]['DTI'].mean()
dti_rejected= df_feat[df_feat['Is_Approved']==0]['DTI'].mean()

print(f'''
  FINDING 1 — Approval Rate
    Overall: {overall:.1f}% of applications are approved.
    {100 - overall:.1f}% of applicants are rejected.

  FINDING 2 — Property Area Impact
    {best_area} has the highest approval rate at {best_rate:.1f}%.
    Likely driven by better income profiles and lower default risk.

  FINDING 3 — Education Effect
    Graduates      : {grad_rate:.1f}% approval rate
    Non-graduates  : {nograd_rate:.1f}% approval rate
    Difference     : {grad_rate - nograd_rate:.1f} percentage points
    (Chi-square confirms this is statistically significant)

  FINDING 4 — Credit History is the Strongest Predictor
    Applicants WITH good history : {cred_yes:.1f}% approved
    Applicants WITHOUT history   : {cred_no:.1f}% approved
    Gap                          : {cred_yes - cred_no:.1f} pp
    → Credit history is the single most decisive variable.

  FINDING 5 — DTI Ratio Confirms Risk Profile
    Avg DTI (Approved) : {dti_approved:.3f}
    Avg DTI (Rejected) : {dti_rejected:.3f}
    → Approved applicants have lower debt relative to income.
''')

print('=' * 65)
print('BUSINESS RECOMMENDATIONS')
print('=' * 65)
print('''
  1. CREDIT HISTORY CAMPAIGNS
     Target applicants without credit history with pre-approval
     products or secured credit cards to build their profile
     before a loan application.

  2. DTI THRESHOLD GUIDELINES
     Implement a DTI threshold of 0.50 as a screening rule.
     Applicants above this level should require additional
     review or a co-applicant with income.

  3. AREA-SPECIFIC RISK PRICING
     Consider differentiated interest rates or limits by
     property area based on observed approval rate differences.

  4. CO-APPLICANT INCENTIVES
     Total income (applicant + co-applicant) shows stronger
     signal than applicant income alone. Incentivize joint
     applications for borderline profiles.
''')

---
## 💾 Step 9 — Export Results

In [ ]:
# ── Export processed dataset ──────────────────────────────────────────────────
df_feat.to_csv('loan_data_processed.csv', index=False)
print('✅ Saved: loan_data_processed.csv')

# ── Export summary by segment ─────────────────────────────────────────────────
summary_report = (df_feat
    .groupby(['Property_Area', 'Education'])
    .agg(
        n               = ('Is_Approved', 'count'),
        approval_rate   = ('Is_Approved', lambda x: round(x.mean() * 100, 1)),
        avg_income      = ('ApplicantIncome', lambda x: round(x.mean(), 0)),
        avg_loan        = ('LoanAmount', lambda x: round(x.mean(), 1)),
        avg_dti         = ('DTI', lambda x: round(x.mean(), 3)),
        avg_risk_score  = ('Risk_Score', lambda x: round(x.mean(), 1))
    )
    .reset_index()
    .sort_values('approval_rate', ascending=False)
)

summary_report.to_csv('loan_summary_by_segment.csv', index=False)
print('✅ Saved: loan_summary_by_segment.csv')

print()
print('Summary by segment:')
summary_report


---

## 📝 Workflow Summary

| Step | Action | Key Output |
|------|--------|------------|
| 1. Setup | Load libraries | Environment ready |
| 2. Load | Read 614 rows from URL | Raw DataFrame |
| 3. Explore | Nulls, types, distributions | Quality baseline |
| 4. Clean | Fill nulls, fix types | Zero-null dataset |
| 5. Engineer | DTI, risk score, binary flags | 10 new features |
| 6. Analyze | Stats + t-test + chi² + correlation | Statistical findings |
| 7. Visualize | 8-chart dashboard | Visual insights |
| 8. Insights | Business recommendations | Actionable findings |
| 9. Export | CSV outputs | Processed data |

**Key skill demonstrated:** Applied Python for BI — taking raw ERP/transactional data from load to insight using pandas, scipy, matplotlib, and seaborn.

---

**Author:** Fredys Caballero · BI Analyst  
**Stack:** Python · pandas · numpy · scipy · matplotlib · seaborn  
**Dataset:** Loan Prediction (dphi-official)